# Koemi MoE Thinking em Google Colab T4

Execute as células em ordem. Este experimento usa o Koemi byte-level com MoE esparso: todos os parâmetros existem no checkpoint, mas cada token passa por um expert. O alvo de aproximadamente 500M totais/20M ativos é medido pelo próprio grafo em `meta`; não é uma promessa de qualidade ou de conclusão em três horas.

O corpus é o subset `default` do `open-r1/OpenR1-Math-220k`, filtrado por completude/correção e limitado para caber no orçamento. Ele ensina raciocínio matemático, não conhecimento geral. O checkpoint e o dataset processado ficam no Google Drive.

In [ ]:
import os, sys, subprocess, time, json, hashlib, re, math
from pathlib import Path

REPO_URL = 'https://github.com/Koemi-AI/Koemi-2OBOV.git'
REPO_REF = 'perf/thinking-training'
REPO_DIR = Path('/content/Koemi-2OBOV')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3.0', 'accelerate>=1.0', 'bitsandbytes>=0.45'], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)
print('ref:', REPO_REF)
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/koemi_runs/moe_thinking_t4')
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = RUN_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / 'koemi-moe-thinking.pt'
print('run directory:', RUN_DIR)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU no Runtime > Change runtime type antes de continuar.')
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print({'gpu': GPU, 'vram_gb': round(VRAM_GB, 2), 'cuda': torch.version.cuda, 'torch': torch.__version__, 'bf16': torch.cuda.is_bf16_supported()})
if VRAM_GB < 12:
    raise RuntimeError('Esta configuração foi dimensionada para uma GPU com pelo menos 12 GB de VRAM.')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

## Planejamento de capacidade
A célula testa candidatos com o modelo em `meta`, sem alocar os 500M pesos. Se o melhor candidato não estiver próximo, reduza `EXPERT_COUNT` ou `TARGET_TOTAL_PARAMS` antes de alocar a GPU.

In [ ]:
from koemi.configuration.settings import ModelSettings
from koemi.model.network import KoemiModel

TARGET_TOTAL_PARAMS = 500_000_000
TARGET_ACTIVE_PARAMS = 20_000_000
EXPERT_COUNT = 64
MEMORY_FEATURES = 64
LOCAL_MEMORY_SIZE = 32
SCAN_CHUNK = 128

def count_params(settings):
    with torch.device('meta'):
        candidate = KoemiModel(settings)
    total = sum(parameter.numel() for parameter in candidate.parameters())
    experts = sum(parameter.numel() for parameter in candidate.experts.experts.parameters())
    one_expert = sum(parameter.numel() for parameter in candidate.experts.experts[0].parameters()) if settings.expert_count else 0
    active = total - experts + one_expert
    return total, active

plans = []
for width in range(768, 1409, 64):
    settings = ModelSettings(embedding_size=width, memory_features=MEMORY_FEATURES, local_memory_size=LOCAL_MEMORY_SIZE, expert_count=EXPERT_COUNT, scan_chunk=SCAN_CHUNK)
    total, active = count_params(settings)
    score = abs(total - TARGET_TOTAL_PARAMS) / TARGET_TOTAL_PARAMS + abs(active - TARGET_ACTIVE_PARAMS) / TARGET_ACTIVE_PARAMS
    plans.append((score, width, total, active, settings))
plans.sort(key=lambda row: row[0])
for _, width, total, active, _ in plans[:5]:
    print(f'width={width:4d} total={total/1e6:8.2f}M active={active/1e6:7.2f}M')
_, WIDTH, TOTAL_PARAMS, ACTIVE_PARAMS, MODEL_SETTINGS = plans[0]
print('selected:', MODEL_SETTINGS)
print(f'total={TOTAL_PARAMS:,} active={ACTIVE_PARAMS:,}')
if TOTAL_PARAMS > 650_000_000 or ACTIVE_PARAMS > 35_000_000:
    raise RuntimeError('O candidato selecionado excede o envelope prudente da T4; reduza a largura ou o número de experts.')

In [ ]:
from datasets import load_dataset

DATASET_ID = 'open-r1/OpenR1-Math-220k'
DATASET_CONFIG = 'default'
MAX_EXAMPLES = 6_000
MAX_TRACE_CHARS = 2_048
VALIDATION_FRACTION = 0.05
DATA_SEED = 17
TRAIN_JSONL = DATA_DIR / 'train.jsonl'
VALID_JSONL = DATA_DIR / 'validation.jsonl'

def first_correct(row):
    complete = row.get('is_reasoning_complete') or []
    verified = row.get('correctness_math_verify') or []
    if isinstance(complete, list) and complete and not any(bool(x) for x in complete):
        return False
    if isinstance(verified, list) and verified and not any(bool(x) for x in verified):
        return False
    return int(row.get('correctness_count') or 0) > 0

def normalize_trace(value):
    if not isinstance(value, str):
        return ''
    value = value.strip()
    if '</think>' in value:
        value = value.split('</think>', 1)[0]
    if '<think>' in value:
        value = value.split('<think>', 1)[1]
    if len(value) <= MAX_TRACE_CHARS:
        return value
    return value[:MAX_TRACE_CHARS - 512].rstrip() + '\n[reasoning truncated]\n' + value[-500:].lstrip()

def stable_validation(identifier):
    digest = hashlib.sha256(identifier.encode('utf-8')).hexdigest()
    return int(digest[:8], 16) % 100 < round(VALIDATION_FRACTION * 100)

stream = load_dataset(DATASET_ID, DATASET_CONFIG, split='train', streaming=True)
train_rows, valid_rows = [], []
seen = 0
for row in stream:
    if not first_correct(row):
        continue
    problem = row.get('problem')
    answer = row.get('answer')
    solution = row.get('solution')
    if not isinstance(problem, str) or not problem.strip() or not isinstance(answer, str) or not answer.strip():
        continue
    trace = normalize_trace(solution)
    if not trace:
        messages = row.get('messages') or []
        if isinstance(messages, list) and messages:
            trace = normalize_trace(messages[-1].get('content', '') if isinstance(messages[-1], dict) else '')
    if not trace:
        continue
    identifier = str(row.get('uuid') or f'openr1-{seen}')
    record = {'id': identifier, 'input': problem.strip(), 'thinking': trace, 'output': answer.strip(), 'metadata': {'dataset': DATASET_ID, 'config': DATASET_CONFIG, 'source': row.get('source', '')}}
    (valid_rows if stable_validation(identifier) else train_rows).append(record)
    seen += 1
    if seen >= MAX_EXAMPLES:
        break
if len(train_rows) < 2 or not valid_rows:
    raise RuntimeError(f'Dataset insuficiente: train={len(train_rows)} validation={len(valid_rows)}')
for path, rows in ((TRAIN_JSONL, train_rows), (VALID_JSONL, valid_rows)):
    with path.open('w', encoding='utf-8') as handle:
        for record in rows:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
print({'train': len(train_rows), 'validation': len(valid_rows), 'train_file_mb': round(TRAIN_JSONL.stat().st_size / 2**20, 2)})

In [ ]:
from koemi.data.readers import load_dataset_records
from koemi.training.dataset import CausalByteDataset, create_training_loader

train_records = load_dataset_records([TRAIN_JSONL], 'canonical').records
valid_records = load_dataset_records([VALID_JSONL], 'canonical').records
SEQUENCE_LENGTH = 256
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
train_dataset = CausalByteDataset(train_records, SEQUENCE_LENGTH)
valid_dataset = CausalByteDataset(valid_records, SEQUENCE_LENGTH)
loader = create_training_loader(train_dataset, BATCH_SIZE, generator=torch.Generator().manual_seed(DATA_SEED), num_workers=2, pin_memory=True, shuffle=True)
valid_loader = create_training_loader(valid_dataset, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print({'train_records': len(train_records), 'train_chunks': len(train_dataset), 'valid_records': len(valid_records), 'valid_chunks': len(valid_dataset), 'sequence_length': SEQUENCE_LENGTH})
if len(train_dataset) < 100:
    raise RuntimeError('Poucos chunks para um treino útil; aumente MAX_EXAMPLES ou MAX_TRACE_CHARS.')

## Treino limitado por relógio
O otimizador 8-bit reduz o estado do Adam; se `bitsandbytes` falhar, a célula aborta em vez de ocultar uma troca de memória que pode estourar a T4. O checkpoint salvo é de pesos; o estado do otimizador não é retomável nesta versão.

In [ ]:
from koemi.data.tokenizer import ByteTokenizer
from koemi.model.execution import ExecutionMode
from koemi.training.checkpoints import CheckpointStore
from koemi.training.objective import calculate_training_objective
from koemi.training.trainer import Trainer

DEVICE = torch.device('cuda')
PRECISION = 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'
LEARNING_RATE = 2e-4
THINKING_LOSS_WEIGHT = 1.25
WEIGHT_DECAY = 0.1
MAX_TRAIN_SECONDS = 3 * 60 * 60 - 600
CHECKPOINT_EVERY_STEPS = 100
LOG_EVERY_STEPS = 10

try:
    import bitsandbytes as bnb
except Exception as error:
    raise RuntimeError('bitsandbytes é obrigatório para este plano de 500M na T4.') from error

model = KoemiModel(MODEL_SETTINGS).to(DEVICE)
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=PRECISION == 'fp16')
autocast_dtype = torch.bfloat16 if PRECISION == 'bf16' else torch.float16
store = CheckpointStore()
model.train()
optimizer.zero_grad(set_to_none=True)
started = time.perf_counter()
step = 0
batch_in_accumulation = 0
running_loss = 0.0
running_answer = 0.0
last_checkpoint = None

while time.perf_counter() - started < MAX_TRAIN_SECONDS:
    for batch in loader:
        if time.perf_counter() - started >= MAX_TRAIN_SECONDS:
            break
        supervised = int((batch['target_ids'] != -100).sum().item())
        thinking = int(((batch['target_ids'] != -100) & batch['thinking_mask']).sum().item())
        if supervised == 0:
            continue
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        target_ids = batch['target_ids'].to(DEVICE, non_blocking=True)
        thinking_mask = batch['thinking_mask'].to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=autocast_dtype):
            output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
            objective = calculate_training_objective(output, target_ids, thinking_mask, THINKING_LOSS_WEIGHT, supervised_token_count=supervised, thinking_token_count=thinking)
            loss = objective.total_loss / GRADIENT_ACCUMULATION
        scaler.scale(loss).backward()
        batch_in_accumulation += 1
        running_loss += float(objective.total_loss.detach())
        running_answer += float(objective.answer_loss.detach())
        if batch_in_accumulation < GRADIENT_ACCUMULATION:
            continue
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        batch_in_accumulation = 0
        step += 1
        if step % LOG_EVERY_STEPS == 0:
            elapsed = time.perf_counter() - started
            print(f'step={step} hours={elapsed/3600:.2f} loss={running_loss/LOG_EVERY_STEPS:.4f} answer_bpb={(running_answer/LOG_EVERY_STEPS)/math.log(2):.4f}')
            running_loss = 0.0
            running_answer = 0.0
        if step and step % CHECKPOINT_EVERY_STEPS == 0:
            last_checkpoint = store.save(CHECKPOINT_PATH, model, overwrite=True)
            print('checkpoint:', last_checkpoint)
    else:
        continue
    break
if batch_in_accumulation:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
if last_checkpoint is None or not CHECKPOINT_PATH.exists():
    last_checkpoint = store.save(CHECKPOINT_PATH, model, overwrite=True)
print({'steps': step, 'elapsed_hours': round((time.perf_counter() - started) / 3600, 3), 'checkpoint': str(last_checkpoint)})

In [ ]:
from koemi.training.trainer import MetricAccumulator

def evaluate_limited(model, data_loader, maximum_batches=64):
    metrics = MetricAccumulator()
    model.eval()
    with torch.inference_mode():
        for batch_index, batch in enumerate(data_loader):
            if batch_index >= maximum_batches:
                break
            supervised = int((batch['target_ids'] != -100).sum().item())
            thinking = int(((batch['target_ids'] != -100) & batch['thinking_mask']).sum().item())
            if supervised == 0:
                continue
            input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
            target_ids = batch['target_ids'].to(DEVICE, non_blocking=True)
            thinking_mask = batch['thinking_mask'].to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', dtype=autocast_dtype):
                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
                objective = calculate_training_objective(output, target_ids, thinking_mask, THINKING_LOSS_WEIGHT, supervised_token_count=supervised, thinking_token_count=thinking)
            metrics.add(output, objective, supervised, thinking)
    model.train()
    return metrics

metrics = evaluate_limited(model, valid_loader)
print({'validation_loss': metrics.mean_loss, 'thinking_loss': metrics.mean_thinking_loss, 'answer_loss': metrics.mean_answer_loss, 'answer_bpb': (metrics.mean_answer_loss / math.log(2)) if metrics.mean_answer_loss is not None else None, 'tokens': metrics.token_count, 'expert_activations': metrics.expert_activation_counts})

In [ ]:
from koemi.data.tokenizer import ByteTokenizer
from koemi.training.generation import generate_text

tokenizer = ByteTokenizer()
question = 'Solve this problem and explain the reasoning: If x + 7 = 19, what is x?'
prompt = '<|input|>\n' + question + '\n<|thinking|>\n'
sample = generate_text(model, tokenizer, prompt, max_new_bytes=768, temperature=0.75, device=str(DEVICE))
print(sample)
print('\ncheckpoint:', CHECKPOINT_PATH)

## Critério de leitura
Compare `validation answer_bpb`, `thinking_loss`, distribuição de `expert_activations` e a amostra. Em byte-level, BPB alto depois de três horas é esperado; não compare esse número diretamente com um modelo BPE. Se a distribuição de experts ficar concentrada em poucos experts, o gargalo é o roteamento hash atual, não o dataset.